In [1]:
import polars as ps

## Jobs sample data - first exploratory analysis

- Files format: jsonl (ndjson)
- Total size of files: 25Gb
- Expected number of lines: 1.5 millions
- Number of columns: 41

-----------------------------

Importing the data directly in Polars do not organize the columns how I want them, I specify a list of columns and will use it to reorganize the lazyframe.

In [2]:
list_col_names = ['public_slug',
 'source',
 'title',
 'company',
 'company_slug',
 'location',
 'description',
 'url',
 'posted_at',
 'created_at',
 'updated_at',
 'closed_at',
 'countries',
 'regions',
 'work_mode',
 'skills',
 'cities',
 'collections',
 'is_tech',
 'seniority',
 'category',
 'posting_language',
 'employment_type',
 'education_level',
 'english_level',
 'experience_years_min',
 'salary_min',
 'salary_max',
 'salary_currency',
 'salary_period',
 'company_name',
 'company_tagline',
 'company_industries',
 'company_hq_country',
 'company_types',
 'company_sizes',
 'domains',
 'yc_batch',
 'yc_status',
 'yc_stage',
 'yc_flags']

Reads the data and infer the potential type while ignoring the errors snce we keep the null. 
Uses '.select' to reorganize the columns. We could also specify a schema in the '.read_ndjson', that would require me to specify a type for every column or would convert everything to String.

In [20]:
lf_jobs = ps.scan_ndjson(source="./sample_jobs.jsonl", ignore_errors=True)

In [21]:
lf_jobs = lf_jobs.select(list_col_names)

Sample consists of 1000 rows. We see the following during a first epxloratory analysis:
- Some columns such as 'cities', 'skills', are lists of values and often contain Nulls. These lists will be used to create dimensions.
- The last 4 columns ('yc_*') are not very useful, too many Nulls (YC is a company repository, but most companies don't have their data there)
- We have a few postings that are not written in English. We will need to get rid of the postings that are not written in Latin Alphabet so we can keep the postings that are written in English even though they are for a country whose alphabet is different (ex: China)
- The locations are uneven and difficult to work with. I think we will work on a Country granularity level, Cities will be too difficult. Some locations are Nulls because of data entry /parsing error, but some are also Nulls because they are 'global' jobs. Will need further analysis.
- There are several different currencies used, the data relating to salaries could be interesting to keep, especially since the Stack Overflow Survey has a 'salary per technology' question.


In [22]:
lf_jobs.describe()

statistic,public_slug,source,title,company,company_slug,location,description,url,posted_at,created_at,updated_at,closed_at,countries,regions,work_mode,skills,cities,collections,is_tech,seniority,category,posting_language,employment_type,education_level,english_level,experience_years_min,salary_min,salary_max,salary_currency,salary_period,company_name,company_tagline,company_industries,company_hq_country,company_types,company_sizes,domains,yc_batch,yc_status,yc_stage,yc_flags
str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,str,f64,f64,f64,f64,str,str,str,str,str,str,f64,f64,f64,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""1000""","""1000""","""1000""","""1000""","""1000""","""1000""","""1000""","""1000""","""1000""","""1000""","""1000""","""464""",1000.0,1000.0,"""1000""",1000.0,1000.0,1000.0,344.0,"""1000""","""1000""","""1000""","""1000""","""1000""","""1000""",478.0,33.0,31.0,"""58""","""46""","""725""","""110""",725.0,0.0,725.0,725.0,725.0,725.0,725.0,725.0,725.0
"""null_count""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""536""",0.0,0.0,"""0""",0.0,0.0,0.0,656.0,"""0""","""0""","""0""","""0""","""0""","""0""",522.0,967.0,969.0,"""942""","""954""","""275""","""890""",275.0,1000.0,275.0,275.0,275.0,275.0,275.0,275.0,275.0
"""mean""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.380814,null,null,null,null,null,null,15.845188,31984.606061,39615.516129,null,null,null,null,null,null,null,null,null,null,null,null,null
"""std""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,18.542746,42699.331393,53805.022467,null,null,null,null,null,null,null,null,null,null,null,null,null
"""min""","""1st-cook-remote-camp-northwest…","""ashby""","""#EG Agentic AI Engineer (AI Te…","""""","""""","""""","""&amp;#xa;&amp;#xa;<p><b>We are…","""https://absa.wd3.myworkdayjobs…","""2023-05-07T08:27:49.92+00:00""","""2026-06-23T08:51:46.937327+00:…","""2026-07-20T16:01:00.217546+00:…","""2026-06-25T09:10:26.511164+00:…",null,null,"""""",null,null,null,0.0,"""""","""""","""""","""""","""""","""""",0.0,14.0,14.0,"""BRL""","""hour""","""ABOUTYOUGmbH""","""A unified platform for data an…",null,null,null,null,null,null,null,null,null
"""25%""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,3.0,545.0,3700.0,null,null,null,null,null,null,null,null,null,null,null,null,null
"""50%""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,5.0,5300.0,7000.0,null,null,null,null,null,null,null,null,null,null,null,null,null
"""75%""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,45.0,60000.0,65000.0,null,null,null,null,null,null,null,null,null,null,null,null,null
"""max""","""zhan-lue-de-patonashitsupu-kom…","""workday""","""采购专员 - 中国南沙""","""zrg""","""zrg""","""東京 OR リーモト（日本)""","""You have begun to prove your w…","""https://zoetis.wd5.myworkdayjo…","""2026-08-06T19:17:31+00:00""","""2026-06-23T09:08:26.578071+00:…","""2026-08-07T02:11:29.240434+00:…","""2026-08-07T02:11:29.240434+00:…",null,null,"""remote""",null,null,null,1.0,"""staff""","""technical_writing""","""zh""","""part_time""","""phd""","""c1""",45.0,180000.0,200000.0,"""usd""","""year""","""zrg""","""Zoetis is the world’s leading …",null,null,null,null,null,null,null,null,null


In [23]:
lf_jobs.schema

C:\Users\stgadmin\AppData\Local\Temp\ipykernel_16220\1532875160.py:1: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  lf_jobs.schema


Schema([('public_slug', String),
        ('source', String),
        ('title', String),
        ('company', String),
        ('company_slug', String),
        ('location', String),
        ('description', String),
        ('url', String),
        ('posted_at', String),
        ('created_at', String),
        ('updated_at', String),
        ('closed_at', String),
        ('countries', List(String)),
        ('regions', List(String)),
        ('work_mode', String),
        ('skills', List(String)),
        ('cities', List(String)),
        ('collections', List(Null)),
        ('is_tech', Boolean),
        ('seniority', String),
        ('category', String),
        ('posting_language', String),
        ('employment_type', String),
        ('education_level', String),
        ('english_level', String),
        ('experience_years_min', Int64),
        ('salary_min', Int64),
        ('salary_max', Int64),
        ('salary_currency', String),
        ('salary_period', String),
        ('comp

## Stack Overflow Survey (2025) sample - first exploratory analysis

- Files format: csv
- Total size of files: 700mb across 6 files (6 years)
- Expected number of lines: 300 000
- Number of columns: 172

### SCHEMA IN 2025_schema.csv

### To have context for some columns and see the final survey results: https://survey.stackoverflow.co/2025/

-----------------------------

In [24]:
lf_survey = ps.scan_csv("sample_stackoverflow_2025_survey.csv")
lf_survey.schema

C:\Users\stgadmin\AppData\Local\Temp\ipykernel_16220\4164024076.py:2: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  lf_survey.schema


Schema([('ResponseId', Int64),
        ('MainBranch', String),
        ('Age', String),
        ('EdLevel', String),
        ('Employment', String),
        ('EmploymentAddl', String),
        ('WorkExp', String),
        ('LearnCodeChoose', String),
        ('LearnCode', String),
        ('LearnCodeAI', String),
        ('AILearnHow', String),
        ('YearsCode', String),
        ('DevType', String),
        ('OrgSize', String),
        ('ICorPM', String),
        ('RemoteWork', String),
        ('PurchaseInfluence', String),
        ('TechEndorseIntro', String),
        ('TechEndorse_1', String),
        ('TechEndorse_2', String),
        ('TechEndorse_3', String),
        ('TechEndorse_4', String),
        ('TechEndorse_5', String),
        ('TechEndorse_6', String),
        ('TechEndorse_7', String),
        ('TechEndorse_8', String),
        ('TechEndorse_9', String),
        ('TechEndorse_13', String),
        ('TechEndorse_13_TEXT', String),
        ('TechOppose_1', String),
 

Sample consists of 1000 rows, each being an answer to the survey.

The goal would be to use this data to create a Facts table giving some context to the IT Professional landscape observed in the Jobs data.

Columns to keep:
- multiple columns of 'haveworkedwith', 'wanttoworkwith', '*admired' etc => Strings with multiple values separated by a ';', give information about technologies used
- country, salary, employment
